# 12 · Build a vision transformer from image patches

**Prerequisites:** CNNs, tensor shapes, training loops, and validation.
**Time:** 40–60 minutes with exercises. This notebook runs fully offline.

You will turn images into patch tokens, add a learned class token and
positional embeddings, pass them through a TransformerEncoder, and train a
small classifier. Compare against a CNN with a fixed equal-epoch budget.

This is an architectural learning exercise. The original ViT results rely
on large-scale training; a tiny transformer trained from scratch on digits
is not a substitute for pretrained models or large-dataset evidence.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from course_utils import get_digits_loaders, TinyCNN, fit, evaluate
train_loader, val_loader, test_loader = get_digits_loaders(batch_size=64, image_size=16, seed=42)
images, labels = next(iter(train_loader))
epochs = 1 if SMOKE_TEST else 8
print('Input batch:', tuple(images.shape), '| labels:', tuple(labels.shape))


## 1. Divide an image into patches

We resize 8×8 digits to 16×16 solely to make a 4×4 patch grid easy to inspect;
resizing adds pixels but no new visual information. Patch size 4 produces
`(16/4)² = 16` tokens. Each grayscale patch contains 16 raw values.
`unfold` below exposes those raw patches for learning and visualization.


In [ ]:
patch_size = 4
raw_patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
# [B,C,grid_h,grid_w,patch_h,patch_w] -> [B,num_patches,patch_values]
flat_patches = raw_patches.permute(0, 2, 3, 1, 4, 5).reshape(images.size(0), 16, 16)
print('Raw patch axes:', tuple(raw_patches.shape))
print('Flattened tokens:', tuple(flat_patches.shape))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(images[0, 0], cmap='gray')
axes[0].set_xticks(np.arange(-0.5, 16, 4), minor=True)
axes[0].set_yticks(np.arange(-0.5, 16, 4), minor=True)
axes[0].grid(which='minor', color='tab:red', linewidth=1)
axes[0].set_title(f'Digit {labels[0].item()} with patch boundaries')
axes[1].imshow(flat_patches[0], cmap='gray', aspect='auto')
axes[1].set(xlabel='Value within patch', ylabel='Patch index', title='16 raw patch tokens')
plt.tight_layout(); plt.show()


## 2. Learn an embedding for each patch

A convolution with kernel size and stride equal to the patch size applies
the same learned linear projection to every nonoverlapping patch. Its
`embed_dim` output channels become token features. This is an efficient
equivalent to flattening each patch and applying one Linear layer.


In [ ]:
embed_dim = 48
patch_projection = nn.Conv2d(1, embed_dim, kernel_size=4, stride=4)
patch_map = patch_projection(images)
patch_tokens = patch_map.flatten(2).transpose(1, 2)
print('Patch map [B,D,grid_h,grid_w]:', tuple(patch_map.shape))
print('Tokens [B,N,D]:', tuple(patch_tokens.shape))
assert patch_tokens.shape[1:] == (16, 48)


## 3. Add a class token and position information

Attention compares tokens without inherently knowing their image locations.
Learned positional embeddings identify locations in this fixed-size grid.
A learned class token is prepended; its final representation summarizes
information useful for classification. The sequence now has 17 tokens.


In [ ]:
class_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
positional_embedding = nn.Parameter(torch.zeros(1, 17, embed_dim))
sequence = torch.cat([class_token.expand(images.size(0), -1, -1), patch_tokens], dim=1)
sequence = sequence + positional_embedding
print('Sequence [batch, 1 + patches, features]:', tuple(sequence.shape))


## 4. Inspect the attention calculation

For one attention head, learned projections produce queries Q, keys K, and
values V. `softmax(Q @ K.T / sqrt(head_dim)) @ V` mixes value vectors according
to query-key compatibility. Each attention row sums to one. Multiple heads
learn separate comparisons, followed by an output projection.

This next matrix uses random projections for the concept only. It is not
the trained model's attention and is not an explanation of its predictions.


In [ ]:
head_dim = 12
token_example = sequence[:1].detach()
qkv = nn.Linear(embed_dim, 3 * head_dim)(token_example)
q, k, v = qkv.chunk(3, dim=-1)
attention = (q @ k.transpose(-2, -1) / math.sqrt(head_dim)).softmax(dim=-1)
attended_values = attention @ v
print('Attention:', tuple(attention.shape), '| output:', tuple(attended_values.shape))
print('First three row sums:', attention[0].sum(-1)[:3].detach())
plt.imshow(attention[0].detach(), cmap='viridis')
plt.xlabel('Key token'); plt.ylabel('Query token')
plt.title('Illustrative random single-head attention'); plt.colorbar(); plt.show()


## 5. Assemble the model

Each encoder block combines multi-head attention, residual connections,
normalization, and a feed-forward network. `batch_first=True` keeps shapes
`[B,N,D]`. Two encoder layers with four heads are enough to inspect the
machinery. The class token goes through normalization and a ten-class head.

TransformerEncoder clones the initial layer. We explicitly reinitialize
cloned layers so they do not start with identical projection weights.


In [ ]:
class TinyViT(nn.Module):
    def __init__(self, image_size=16, patch_size=4, dim=48, heads=4, depth=2):
        super().__init__()
        assert image_size % patch_size == 0 and dim % heads == 0
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(1, dim, kernel_size=patch_size, stride=patch_size)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.position = nn.Parameter(torch.zeros(1, self.num_patches + 1, dim))
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads,
            dim_feedforward=dim * 2, dropout=0.1, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 10)
        self.apply(self.initialize)
        # in_proj_weight is a parameter of MultiheadAttention, not a Linear child.
        for block in self.encoder.layers:
            nn.init.xavier_uniform_(block.self_attn.in_proj_weight)
            if block.self_attn.in_proj_bias is not None:
                nn.init.zeros_(block.self_attn.in_proj_bias)
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.position, std=0.02)

    @staticmethod
    def initialize(module):
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight); nn.init.zeros_(module.bias)

    def tokenize(self, x):
        if x.shape[-2:] != (self.image_size, self.image_size):
            raise ValueError('Input size must match the configured positional grid.')
        tokens = self.patch_embed(x).flatten(2).transpose(1, 2)
        return torch.cat([self.cls.expand(x.size(0), -1, -1), tokens], dim=1) + self.position

    def forward(self, x):
        encoded = self.encoder(self.tokenize(x))
        return self.head(self.norm(encoded[:, 0]))

shape_model = TinyViT()
with torch.no_grad():
    print('Embedded sequence:', tuple(shape_model.tokenize(images[:2]).shape))
    print('Classification logits:', tuple(shape_model(images[:2]).shape))


## 6. Compare with a CNN under a declared budget

Both models get the same input resolution, data split, batch size, seed,
optimizer family, learning rate, and epoch budget. Each uses its best
validation-loss checkpoint. Architectures have different parameter counts
and different optimization needs, so equal epochs are an instructional
comparison, not equal compute or equally tuned performance.

A CNN's local structure often helps on small datasets. Expect the transformer
to need more careful tuning and data; it is fine if the CNN wins here.


In [ ]:
models, histories = {}, {}
for name, factory in [('CNN', TinyCNN), ('ViT', TinyViT)]:
    seed_everything(42)
    train, val, _ = get_digits_loaders(batch_size=64, image_size=16, seed=42)
    model = factory().to(device)
    history = fit(model, train, val, epochs=epochs, lr=0.003,
                  weight_decay=1e-4, device=device)
    models[name], histories[name] = model, history
print('Both models trained; test data remain unopened.')


In [ ]:
rows = []
for name, model in models.items():
    result = evaluate(model, val_loader, device=device)
    rows.append({'model': name, 'parameters': sum(p.numel() for p in model.parameters()),
                 'val_loss': result['loss'], 'val_accuracy': result['accuracy']})
validation_comparison = pd.DataFrame(rows)
display(validation_comparison)
for name, history in histories.items():
    table = pd.DataFrame(history)
    plt.plot(table['epoch'], table['val_loss'], 'o-', label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation cross-entropy'); plt.legend(); plt.show()


## 7. Evaluate one preselected model and inspect predictions

Select by validation loss, then open test once. Show fixed test examples
rather than searching for the most impressive successes. Continue exploring
architecture on validation data; reserve new final data for new claims.


In [ ]:
chosen_name = validation_comparison.loc[validation_comparison['val_loss'].idxmin(), 'model']
chosen_model = models[chosen_name]
final = evaluate(chosen_model, test_loader, device=device)
print('Selected:', chosen_name, '| test loss:', final['loss'], '| test accuracy:', final['accuracy'])
test_images, _ = next(iter(test_loader))
probabilities = final['logits'].softmax(1)
confidence, predictions = probabilities.max(1)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(test_images[i, 0], cmap='gray')
    ax.set_title(f'true {final["targets"][i].item()} → {predictions[i].item()}\np={confidence[i]:.2f}')
    ax.axis('off')
plt.tight_layout(); plt.show()
out_dir = ARTIFACTS / '12_vit'; out_dir.mkdir(parents=True, exist_ok=True)
validation_comparison.to_csv(out_dir / 'validation_comparison.csv', index=False)
torch.save({'model_state': models['ViT'].state_dict(),
            'config': {'image_size': 16, 'patch_size': 4, 'dim': 48, 'heads': 4, 'depth': 2}},
           out_dir / 'tiny_vit.pt')
print('Saved learning artifact:', out_dir / 'tiny_vit.pt')


## Practice 1 · Count attention cost

Change patch size and calculate the number of tokens and attention-score
entries per head. Dense attention's score matrix grows with the square of
sequence length. Smaller patches preserve detail but consume more memory.


In [ ]:
cost_rows = []
for patch in [2, 4, 8]:
    tokens = (16 // patch) ** 2 + 1
    cost_rows.append({'patch_size': patch, 'tokens_including_cls': tokens,
                      'attention_entries_per_head': tokens ** 2})
display(pd.DataFrame(cost_rows))


<details><summary>Solution idea</summary>
Patch sizes 2, 4, and 8 yield 65, 17, and 5 tokens. Attention-score matrices
contain 4225, 289, and 25 entries per head per image, respectively. Actual
memory depends on the attention implementation and other activations too.
</details>

## Practice 2 · Remove positional information for one diagnostic

Temporarily zero the learned position tensor and measure **validation** loss.
Restore it immediately. This measures reliance on learned positions in this
model; it is not equivalent to training an architecture without positions.


In [ ]:
vit = models['ViT']
original_position = vit.position.detach().clone()
baseline_val = evaluate(vit, val_loader, device=device)
try:
    with torch.no_grad():
        vit.position.zero_()
    no_position_val = evaluate(vit, val_loader, device=device)
finally:
    with torch.no_grad():
        vit.position.copy_(original_position)
print({'validation_loss_original': baseline_val['loss'],
       'validation_loss_zero_position': no_position_val['loss']})


<details><summary>Solution idea</summary>
Zeroing positions creates an intervention the model was not trained for.
A proper architecture ablation retrains both variants across multiple seeds
with controlled budgets. One validation difference cannot prove causality
about transformers in general.
</details>

## Practice 3 · Change patch size safely

Build a patch-size-2 model, inspect shapes, and verify output dimensions.
Its positional embedding length changes, so it requires newly initialized
parameters (or an explicit positional interpolation method). This starter
performs a forward pass only; optionally train it using validation data.


In [ ]:
alternative = TinyViT(patch_size=2).to(device)
alternative.eval()
with torch.no_grad():
    small_batch = images[:2].to(device)
    print('Alternative sequence:', tuple(alternative.tokenize(small_batch).shape))
    print('Alternative logits:', tuple(alternative(small_batch).shape))
print('Alternative parameters:', sum(p.numel() for p in alternative.parameters()))


<details><summary>Solution idea</summary>
Expect [2,65,48] tokens and [2,10] logits. Patch projection parameters decrease
while positional parameters increase. Training and attention compute can
increase even when total parameter count changes little.
</details>

**Completion check:** trace `[B,1,16,16] → [B,16,48] → [B,17,48] → [B,10]`,
explain class and position tokens, compute attention shapes, and state the
limits of the CNN/ViT comparison. Finishing a small example builds foundations;
expert practice also requires larger datasets, robust evaluation, and iteration.

**Primary sources:** [Dosovitskiy et al., ViT](https://arxiv.org/abs/2010.11929),
[Vaswani et al., attention](https://arxiv.org/abs/1706.03762),
[PyTorch TransformerEncoder](https://docs.pytorch.org/docs/2.11/generated/torch.nn.TransformerEncoder.html).

**Next:** [13 · Explainability and inference](13_explainability_and_inference.ipynb).
